# IEP1A — YOLOv8-seg training and evaluation (Google Colab)

This notebook fine-tunes **Ultralytics YOLOv8n-seg** for **LibraryAI IEP1A**: page **instance segmentation** with classes **`page_left`** (id 0) and page_right (id 1).

## Prerequisites

- **Runtime:** Colab **GPU** (e.g. Tesla T4).
- **Google Drive:** Zips below are copied from `MyDrive/`; change paths if your filenames differ.
- **YOLO layout:** Each dataset root has `data.yaml`, `images/{train,val,test}`, and `labels/{train,val,test}` with matching image/label stems.

## Default training settings (used in this notebook)

| Setting | Value |
|---------|--------|
| Model | `yolov8n-seg.pt` |
| CLI | `yolo task=segment mode=train` / `mode=val` |
| `imgsz` | 640 |
| `batch` | 4 |
| `epochs` | 40 |
| `device` | `0` (GPU) |

Trained weights: `/content/runs/segment/<run_name>/weights/best.pt` and `last.pt`.

## Drive zips referenced here

| File (under `MyDrive`) | Extracted to | Typical use |
|-------------------------|--------------|-------------|
| `datasets.zip` | `/content/datasets` | **Experiment 1 — newspapers:** train + in-domain test; run folder **`train2`**. |
| `book_seg.zip` | `/content/book_seg` | **Experiment 2 — books:** train **`book_only`**; in-domain test on `book_seg` test split. |
| `mixed.zip` | `/content/mixed` | **Experiment 3 — mixed:** train **`mixed2`**; test on `mixed` test split. |
| `book_all.zip` | `/content/book_all` | **Cross-domain test only:** all book pages as test (or a larger book-only eval set). |

## How the notebook is organized

1. **Setup:** Install Ultralytics, verify CUDA.
2. **Load `datasets`:** Mount Drive, unzip, list folders, (re)write `data.yaml` for `/content/datasets`.
3. **Experiment 1 — newspapers:** Train and test on **`datasets`** only (newspaper imagery). Training writes **`train2`**; then `yolo mode=val … split=test` scores **in-domain** test mAP on the same dataset.
4. **Experiment 2 — books:** Load **`book_seg`**, train **`book_only`**, evaluate on **`book_seg`** held-out test (books in-domain).
5. **Experiment 3 — mixed:** Load **`mixed`**, train **`mixed2`**, evaluate on **`mixed`** test (books + newspapers in one split).
6. **Cross-domain (end):** Take the **newspaper-trained** checkpoint **`train2/weights/best.pt`** and run **`mode=val`** with **`book_seg`** or **`book_all`** `data.yaml` and **`split=test`**. That measures how well the model trained only on newspapers **generalizes to all book pages** (no newspaper images in that eval set). Optional cells use `find` to locate `best.pt` / `data.yaml` paths.

## Operational notes

- **`unzip` prompts:** If asked to replace files, answer **`A`** (all) or remove `/content/<folder>` before unzipping for a clean tree.
- **Grayscale images:** Convert to RGB before training (YOLO expects 3 channels).
- **Spec:** IEP1A is **YOLOv8-seg**; **nano** is a valid variant for speed and GPU memory.

## Before disconnecting Colab

Download `best.pt` from `/content/runs/segment/*/weights/` and any plots you need from the corresponding run directories.

### Environment
Install **ultralytics**, import **torch**, print whether **CUDA** is available and the GPU name.


In [ ]:
!pip -q install ultralytics
import torch
print("CUDA:", torch.cuda.is_available(), "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00
CUDA: True GPU: Tesla T4


### Experiment 1 — load newspaper dataset (`datasets`)
Mount **Google Drive**, copy `MyDrive/datasets.zip` to `/content/`, and unzip (creates `/content/datasets/`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/datasets.zip" /content/
!unzip -q /content/datasets.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Optional — list Drive root
List `MyDrive` to confirm `datasets.zip` (and later other zips) are present.


In [ ]:
!ls -lah "/content/drive/MyDrive"

total 1.0G
drwx------ 2 root root 4.0K Dec 12 08:47 'Colab Notebooks'
-rw------- 1 root root 1.0G Mar 31 13:40  datasets.zip


### Optional scratch
Placeholder cell — safe to delete or use for one-off commands.


### Newspaper data — copy, unzip, list
Copy `datasets.zip` again, unzip into `/content`, list `/content` and `/content/datasets` (troubleshooting / clean extract).


In [ ]:
!cp "/content/drive/MyDrive/datasets.zip" /content/
!unzip -q /content/datasets.zip -d /content/
!ls -lah /content
!ls -lah /content/datasets

replace /content/datasets/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
total 1.5G
drwxr-xr-x 1 root root 4.0K Mar 31 15:39 .
drwxr-xr-x 1 root root 4.0K Mar 31 13:34 ..
drwxrwxrwx 4 root root 4.0K Mar 31 15:27 book_seg
-rw------- 1 root root 467M Mar 31 15:39 book_seg.zip
drwxr-xr-x 4 root root 4.0K Mar 23 13:34 .config
drwxrwxrwx 4 root root 4.0K Mar 31 15:40 datasets
-rw------- 1 root root 1.0G Mar 31 15:40 datasets.zip
drwx------ 5 root root 4.0K Mar 31 13:48 drive
drwxr-xr-x 3 root root 4.0K Mar 31 15:00 runs
drwxr-xr-x 1 root root 4.0K Mar 23 13:34 sample_data
-rw-r--r-- 1 root root 5.3M Mar 31 15:02 yolo26n.pt
-rw-r--r-- 1 root root 6.8M Mar 31 15:00 yolov8n-seg.pt
total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 15:40 .
drwxr-xr-x 1 root root 4.0K Mar 31 15:39 ..
-rw-rw-rw- 1 root root  143 Mar 31 11:19 data.yaml
drwxrwxrwx 5 root root 4.0K Mar 31 10:40 images
drwxrwxrwx 5 root root 4.0K Mar 31 15:40 labels


### Inspect `datasets` layout
Show `datasets`, `images/`, and `labels/` with `train` / `val` / `test` subfolders.


In [ ]:
!ls -lah /content/datasets
!ls -lah /content/datasets/images
!ls -lah /content/datasets/labels

total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 14:57 .
drwxr-xr-x 1 root root 4.0K Mar 31 14:22 ..
-rw-rw-rw- 1 root root  143 Mar 31 11:19 data.yaml
drwxrwxrwx 5 root root 4.0K Mar 31 10:40 images
drwxrwxrwx 5 root root 4.0K Mar 31 14:57 labels
total 36K
drwxrwxrwx 5 root root 4.0K Mar 31 10:40 .
drwxrwxrwx 4 root root 4.0K Mar 31 14:57 ..
drwxrwxrwx 2 root root 4.0K Mar 31 14:57 test
drwxrwxrwx 2 root root  20K Mar 31 14:57 train
drwxrwxrwx 2 root root 4.0K Mar 31 14:57 val
total 128K
drwxrwxrwx 5 root root 4.0K Mar 31 14:57 .
drwxrwxrwx 4 root root 4.0K Mar 31 14:57 ..
drwxrwxrwx 2 root root 4.0K Mar 31 14:57 test
drwxrwxrwx 2 root root  20K Mar 31 14:57 train
-rw-rw-rw- 1 root root  72K Mar 31 13:19 train.cache
drwxrwxrwx 2 root root 4.0K Mar 31 14:57 val
-rw-rw-rw- 1 root root  19K Mar 31 13:19 val.cache


### Write `/content/datasets/data.yaml`
Define YOLO paths (`path`, `train`, `val`, `test`) and class names `page_left` / `page_right`.


In [ ]:
yaml_text = """path: /content/datasets
train: images/train
val: images/val
test: images/test

names:
  0: page_left
  1: page_right
"""
open("/content/datasets/data.yaml", "w").write(yaml_text)
print(open("/content/datasets/data.yaml").read())

path: /content/datasets
train: images/train
val: images/val
test: images/test

names:
  0: page_left
  1: page_right



### Sanity check — split folders
List `images` and `labels` to confirm `test`, `train`, `val` exist.


In [ ]:
!ls /content/datasets/images
!ls /content/datasets/labels

test  train  val
test  train  train.cache  val  val.cache


### Experiment 1 — **train** (newspapers)
Fine-tune **YOLOv8n-seg** on `/content/datasets`. Default run folder: **`train2`** (see Colab output for exact `save_dir`).


In [ ]:
!yolo task=segment mode=train model=yolov8n-seg.pt data=/content/datasets/data.yaml imgsz=640 batch=4 epochs=40 device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

### Experiment 1 — **test** (newspapers, in-domain)
Run `yolo mode=val` with **`train2/weights/best.pt`**, same `data.yaml`, **`split=test`** — metrics on held-out newspaper images.


In [ ]:
!yolo task=segment mode=val model=/content/runs/segment/train2/weights/best.pt data=/content/datasets/data.yaml split=test device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1726.5±264.2 MB/s, size: 3554.6 KB)
val: Scanning /content/datasets/labels/test... 50 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 50/50 793.8it/s 0.1s
val: New cache created: /content/datasets/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.8s/it 7.4s
                   all         50         99          1      0.999      0.995      0.985          1      0.999      0.995      0.989
             page_left         49         49          1      0.999      0.995      0.993          1      0.999      0.995      0.993
            page_right         50         50          1      0.999      0.995      0.977          1      0.999      0.995     

### Experiment 2 — load book split (`book_seg`)
Mount Drive (if needed), copy `book_seg.zip`, unzip to `/content/book_seg`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/book_seg.zip" /content/
!unzip -q /content/book_seg.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Optional — list Drive (book zip)
Confirm `book_seg.zip` on Drive.


In [ ]:
!ls -lah "/content/drive/MyDrive"

total 1.5G
-rw------- 1 root root 467M Mar 31 15:32  book_seg.zip
drwx------ 2 root root 4.0K Dec 12 08:47 'Colab Notebooks'
-rw------- 1 root root 1.0G Mar 31 13:40  datasets.zip


### Unzip `book_seg`
Copy zip to `/content`, unzip, list `/content` and `/content/book_seg`.


In [ ]:
!cp "/content/drive/MyDrive/book_seg.zip" /content/
!unzip -q /content/book_seg.zip -d /content/
!ls -lah /content
!ls -lah /content/book_seg

replace /content/book_seg/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
total 1.5G
drwxr-xr-x 1 root root 4.0K Mar 31 15:39 .
drwxr-xr-x 1 root root 4.0K Mar 31 13:34 ..
drwxrwxrwx 4 root root 4.0K Mar 31 15:41 book_seg
-rw------- 1 root root 467M Mar 31 15:41 book_seg.zip
drwxr-xr-x 4 root root 4.0K Mar 23 13:34 .config
drwxrwxrwx 4 root root 4.0K Mar 31 15:40 datasets
-rw------- 1 root root 1.0G Mar 31 15:40 datasets.zip
drwx------ 5 root root 4.0K Mar 31 13:48 drive
drwxr-xr-x 3 root root 4.0K Mar 31 15:00 runs
drwxr-xr-x 1 root root 4.0K Mar 23 13:34 sample_data
-rw-r--r-- 1 root root 5.3M Mar 31 15:02 yolo26n.pt
-rw-r--r-- 1 root root 6.8M Mar 31 15:00 yolov8n-seg.pt
total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 15:41 .
drwxr-xr-x 1 root root 4.0K Mar 31 15:39 ..
-rw-rw-rw- 1 root root  116 Mar 31 15:30 data.yaml
drwxrwxrwx 5 root root 4.0K Mar 31 14:19 images
drwxrwxrwx 5 root root 4.0K Mar 31 14:20 labels


### Experiment 2 — **train** (books only)
Train with `data=/content/book_seg/data.yaml`, run name **`book_only`**. Weights: `/content/runs/segment/book_only/weights/`.


In [ ]:
!yolo task=segment mode=train model=yolov8n-seg.pt data=/content/book_seg/data.yaml imgsz=640 batch=4 epochs=40 device=0 name=book_only

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/book_seg/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=book_only, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspecti

### Experiment 2 — **test** (books, in-domain)
Evaluate **`book_only/weights/best.pt`** on **`book_seg`** test split.


In [ ]:
!yolo task=segment mode=val model=/content/runs/segment/book_only/weights/best.pt data=/content/book_seg/data.yaml split=test device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1333.7±82.6 MB/s, size: 7840.4 KB)
val: Scanning /content/book_seg/labels/test... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 209.3it/s 0.0s
val: New cache created: /content/book_seg/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.3s/it 1.3s
                   all         10         20      0.987          1      0.995      0.995      0.987          1      0.995      0.995
             page_left         10         10      0.995          1      0.995      0.995      0.995          1      0.995      0.995
            page_right         10         10      0.979          1      0.995      0.995      0.979          1      0.995      

### Experiment 3 — load mixed dataset
Mount Drive, copy `mixed.zip`, unzip (creates `/content/mixed`).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/mixed.zip" /content/
!unzip -q /content/mixed.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cp: cannot stat '/content/drive/MyDrive/mixed.zip': No such file or directory
replace /content/mixed/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


### Optional — list Drive (`mixed.zip`)
Confirm `mixed.zip` is on Drive.


In [ ]:
!ls -lah "/content/drive/MyDrive"

total 3.0G
-rw------- 1 root root 467M Mar 31 15:32  book_seg.zip
drwx------ 2 root root 4.0K Dec 12 08:47 'Colab Notebooks'
-rw------- 1 root root 1.0G Mar 31 13:40  datasets.zip
-rw------- 1 root root 1.5G Mar 31 16:43  mixed.zip


### Optional scratch
Placeholder — safe to delete.


### Unzip `mixed`
Copy `mixed.zip` to `/content`, unzip, list `/content/mixed`.


In [ ]:
!cp "/content/drive/MyDrive/mixed.zip" /content/
!unzip -q /content/mixed.zip -d /content/
!ls -lah /content
!ls -lah /content/mixed

replace /content/mixed/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
total 3.0G
drwxr-xr-x 1 root root 4.0K Mar 31 16:31 .
drwxr-xr-x 1 root root 4.0K Mar 31 13:34 ..
drwxrwxrwx 4 root root 4.0K Mar 31 15:41 book_seg
-rw------- 1 root root 467M Mar 31 16:32 book_seg.zip
drwxr-xr-x 4 root root 4.0K Mar 23 13:34 .config
drwxrwxrwx 4 root root 4.0K Mar 31 15:40 datasets
-rw------- 1 root root 1.0G Mar 31 15:40 datasets.zip
drwx------ 5 root root 4.0K Mar 31 13:48 drive
drwxrwxrwx 4 root root 4.0K Mar 31 17:02 mixed
-rw------- 1 root root 1.5G Mar 31 17:02 mixed.zip
drwxr-xr-x 3 root root 4.0K Mar 31 15:00 runs
drwxr-xr-x 1 root root 4.0K Mar 23 13:34 sample_data
-rw-r--r-- 1 root root 5.3M Mar 31 15:02 yolo26n.pt
-rw-r--r-- 1 root root 6.8M Mar 31 15:00 yolov8n-seg.pt
total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 17:02 .
drwxr-xr-x 1 root root 4.0K Mar 31 16:31 ..
-rw-r--r-- 1 root root  113 Mar 31  2026 data.yaml
drwxrwxrwx 5 root root 4.0K Mar 31 15:57 images
drwxrwxrwx 5 root root 

### Experiment 3 — **train** (mixed books + newspapers)
Fine-tune on `/content/mixed/data.yaml`. Run folder **`mixed2`** (Ultralytics may increment if `mixed` already exists).


In [ ]:
!yolo task=segment mode=train model=yolov8n-seg.pt data=/content/mixed/data.yaml imgsz=640 batch=4 epochs=40 device=0 name=mixed

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/mixed/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mixed2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0

### Experiment 3 — **test** (mixed, in-domain)
Evaluate **`mixed2/weights/best.pt`** on **`mixed`** test split.


In [ ]:
!yolo task=segment mode=val model=/content/runs/segment/mixed2/weights/best.pt data=/content/mixed/data.yaml split=test device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 580.9±357.9 MB/s, size: 4418.1 KB)
val: Scanning /content/mixed/labels/test... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 331.2it/s 0.2s
val: New cache created: /content/mixed/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.3s/it 9.3s
                   all         60        119      0.993      0.999      0.995      0.987      0.993      0.999      0.995      0.992
             page_left         59         59          1      0.999      0.995      0.994          1      0.999      0.995      0.994
            page_right         60         60      0.985          1      0.995      0.981      0.985          1      0.995      0.991


### Find trained checkpoints
`find` all **`best.pt`** under `/content/runs/segment`.


In [ ]:
!find /content/runs/segment -name best.pt

/content/runs/segment/mixed/weights/best.pt
/content/runs/segment/train2/weights/best.pt
/content/runs/segment/book_only/weights/best.pt
/content/runs/segment/mixed2/weights/best.pt


### Find `data.yaml` and book folders
Locate yaml files and directories matching book-related names (debugging paths).


In [ ]:
!find /content -maxdepth 3 -name "data.yaml"
!find /content -maxdepth 3 -type d | grep -E "book|books|book_all"

/content/mixed/data.yaml
/content/datasets/data.yaml
/content/book_seg/data.yaml
/content/drive/MyDrive/Colab Notebooks
/content/book_seg
/content/book_seg/labels
/content/book_seg/labels/test
/content/book_seg/labels/val
/content/book_seg/labels/train
/content/book_seg/images
/content/book_seg/images/test
/content/book_seg/images/val
/content/book_seg/images/train
/content/runs/segment/book_only


### Cross-domain — load `book_all`
Mount Drive, copy `book_all.zip` (all book images for evaluation).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp "/content/drive/MyDrive/book_all.zip" /content/
!unzip -q /content/book_all.zip -d /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Unzip `book_all`
Extract to `/content/book_all` and list structure.


In [ ]:
!cp "/content/drive/MyDrive/book_all.zip" /content/
!unzip -q /content/book_all.zip -d /content/
!ls -lah /content
!ls -lah /content/book_all

replace /content/book_all/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
total 3.4G
drwxr-xr-x 1 root root 4.0K Mar 31 17:54 .
drwxr-xr-x 1 root root 4.0K Mar 31 13:34 ..
drwxrwxrwx 4 root root 4.0K Mar 31 17:55 book_all
-rw------- 1 root root 467M Mar 31 17:55 book_all.zip
drwxrwxrwx 4 root root 4.0K Mar 31 15:41 book_seg
-rw------- 1 root root 467M Mar 31 16:32 book_seg.zip
drwxr-xr-x 4 root root 4.0K Mar 23 13:34 .config
drwxrwxrwx 4 root root 4.0K Mar 31 15:40 datasets
-rw------- 1 root root 1.0G Mar 31 15:40 datasets.zip
drwx------ 5 root root 4.0K Mar 31 13:48 drive
drwxrwxrwx 4 root root 4.0K Mar 31 17:54 mixed
-rw------- 1 root root 1.5G Mar 31 17:54 mixed.zip
drwxr-xr-x 3 root root 4.0K Mar 31 15:00 runs
drwxr-xr-x 1 root root 4.0K Mar 23 13:34 sample_data
-rw-r--r-- 1 root root 5.3M Mar 31 15:02 yolo26n.pt
-rw-r--r-- 1 root root 6.8M Mar 31 15:00 yolov8n-seg.pt
total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 17:55 .
drwxr-xr-x 1 root root 4.0K Mar 31 17:54 ..
-rw-rw-rw- 1 ro

### Cross-domain — **test** `train2` on **`book_seg`** test
Newspaper-trained **`train2/weights/best.pt`** evaluated on book **`book_seg`** test split — measures domain gap (papers → books).


In [ ]:
!yolo task=segment mode=val model=/content/runs/segment/train2/weights/best.pt data=/content/book_seg/data.yaml split=test device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1569.3±1070.9 MB/s, size: 7812.7 KB)
val: Scanning /content/book_seg/labels/test.cache... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 2.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1s/it 1.1s
                   all         10         20      0.785        0.5      0.587      0.496      0.728       0.45      0.546      0.473
             page_left         10         10          1      0.499      0.582      0.498          1      0.499      0.582      0.511
            page_right         10         10       0.57        0.5      0.591      0.494      0.456        0.4      0.509      0.435
Speed: 1.7ms preprocess, 19.3ms inference, 0.0m

### Optional — inspect `book_all` tree
List `book_all` images/labels (e.g. test-only layout).


In [ ]:
!find /content -maxdepth 3 -type d | grep -i "book_all"
!find /content -maxdepth 4 -name "data.yaml" | grep -i "book_all"
!ls -lah /content/book_all
!ls -lah /content/book_all/images
!ls -lah /content/book_all/labels

/content/book_all
/content/book_all/labels
/content/book_all/labels/test
/content/book_all/images
/content/book_all/images/test
/content/book_all/data.yaml
total 20K
drwxrwxrwx 4 root root 4.0K Mar 31 17:55 .
drwxr-xr-x 1 root root 4.0K Mar 31 17:54 ..
-rw-rw-rw- 1 root root  116 Mar 31 17:29 data.yaml
drwxrwxrwx 3 root root 4.0K Mar 31 17:28 images
drwxrwxrwx 3 root root 4.0K Mar 31 17:28 labels
total 12K
drwxrwxrwx 3 root root 4.0K Mar 31 17:28 .
drwxrwxrwx 4 root root 4.0K Mar 31 17:55 ..
drwxrwxrwx 2 root root 4.0K Mar 31 17:55 test
total 12K
drwxrwxrwx 3 root root 4.0K Mar 31 17:28 .
drwxrwxrwx 4 root root 4.0K Mar 31 17:55 ..
drwxrwxrwx 2 root root 4.0K Mar 31 17:55 test


### Cross-domain — **test** `train2` on **`book_all`**
Same newspaper-trained checkpoint evaluated on **all** book pages prepared in `book_all` (typically test split) — broader book-only stress test.


In [ ]:
!yolo task=segment mode=val model=/content/runs/segment/train2/weights/best.pt data=/content/book_all/data.yaml split=test device=0

Ultralytics 8.4.32 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-seg summary (fused): 86 layers, 3,258,454 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 207.5±15.9 MB/s, size: 7697.9 KB)
val: Scanning /content/book_all/labels/test... 63 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 63/63 350.5it/s 0.2s
val: New cache created: /content/book_all/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 2.4s/it 9.5s
                   all         63        123      0.752      0.485      0.499      0.401      0.715      0.454      0.467      0.389
             page_left         63         63      0.936      0.466      0.563      0.461      0.937      0.468      0.561      0.466
            page_right         60         60      0.568      0.503      0.434      0.342      0.494      0.439      0.373      0